# Notebook 5 – OCR : textes imprimés et manuscrits
**Projet Computer Vision IG.2405 – 2026**

Ce notebook couvre la lecture des textes par OCR :
1. **Textes imprimés** : bande CODES EXAM (Module, Professeur, Date, Code)
2. **Textes manuscrits** : Prénom et Nom (grilles de cellules)
3. **Chiffres imprimés** : Note maximale, Note pour valider
4. **Chiffres manuscrits** : Mantisse et Exposant des réponses numériques

> Backend OCR : **EasyOCR** (Deep Learning – ResNet + LSTM), justifié car les textes
> imprimés et manuscrits tombent dans la catégorie 'haut niveau' autorisée par le cahier des charges.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from utils.grid_decoder import normalize_page, get_roi, ROI_CODES_EXAM, ROI_FIRSTNAME, ROI_NAME
from utils.form_aligner import deskew
from utils.ocr_utils import (
    ocr_codes_exam, ocr_text, ocr_number,
    ocr_handwritten_mantisse, ocr_handwritten_exposant, ocr_handwritten_unite,
    _upscale_binarize, _ocr_raw, _to_gray,
)
from utils.pdf_utils import pdf_to_images

FORM_DIR = os.path.join('PROJECT 2026 -DATABASE-20260518', 'FORM1')

pdf_files = sorted([f for f in os.listdir(FORM_DIR) if f.lower().endswith('.pdf')])
img_files = sorted([f for f in os.listdir(FORM_DIR)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
print(f'{len(pdf_files)} PDFs, {len(img_files)} images')

## 1. Lecture de la bande CODES EXAM (texte imprimé)

La bande colorée en haut de page contient : **Module | Professeur | Date | Code**

Pipeline : agrandissement × 3 → binarisation → EasyOCR → parsing par regex

In [ ]:
# Charger un PDF propre pour l'OCR (meilleure qualité)
if pdf_files:
    sample_pdf = os.path.join(FORM_DIR, pdf_files[0])
    pages = pdf_to_images(sample_pdf, dpi=150)
    pdf_page1 = normalize_page(pages[0])

    roi_codes = get_roi(pdf_page1, ROI_CODES_EXAM)

    # Prétraitement visualisé
    processed = _upscale_binarize(roi_codes, scale=3)

    fig, axes = plt.subplots(1, 2, figsize=(14, 3))
    axes[0].imshow(cv2.cvtColor(roi_codes, cv2.COLOR_BGR2RGB))
    axes[0].set_title('ROI CODES EXAM brut')
    axes[0].axis('off')

    axes[1].imshow(processed, cmap='gray')
    axes[1].set_title('Après agrandissement ×3 + binarisation Otsu')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

    # OCR
    codes = ocr_codes_exam(roi_codes)
    print('Résultats OCR CODES EXAM :')
    for k, v in codes.items():
        print(f'  {k:12s} : "{v}"')

## 2. Lecture des notes (chiffres imprimés)

La Note maximale et la Note pour valider sont des chiffres imprimés dans des cases encadrées.

Approche : EasyOCR avec allowlist `0123456789` + fallback segmentation bas niveau.

In [ ]:
from utils.grid_decoder import (
    ROI_NOTE_MAX, ROI_NOTE_VALID, ROI_NOTES_COMBINED,
    read_note_maximale, read_note_pour_valider
)

if pdf_files:
    roi_notes = get_roi(pdf_page1, ROI_NOTES_COMBINED)

    fig, axes = plt.subplots(1, 3, figsize=(10, 3))
    axes[0].imshow(cv2.cvtColor(get_roi(pdf_page1, ROI_NOTE_MAX), cv2.COLOR_BGR2RGB))
    axes[0].set_title('ROI Note max')
    axes[0].axis('off')

    axes[1].imshow(cv2.cvtColor(get_roi(pdf_page1, ROI_NOTE_VALID), cv2.COLOR_BGR2RGB))
    axes[1].set_title('ROI Note valider')
    axes[1].axis('off')

    axes[2].imshow(cv2.cvtColor(roi_notes, cv2.COLOR_BGR2RGB))
    axes[2].set_title('ROI combinée')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

    note_max   = read_note_maximale(pdf_page1)
    note_valid = read_note_pour_valider(pdf_page1)
    print(f'Note maximale   : {note_max}')
    print(f'Note pour valid : {note_valid}')

## 3. Lecture du Prénom et du Nom (texte manuscrit)

Prénom et Nom sont écrits **lettre par lettre** dans des cellules individuelles.

Algorithme :
1. Binarisation → projection verticale
2. Détection des **séparateurs inter-cellules** (colonnes à forte densité d'encre)
3. OCR cellule par cellule (EasyOCR, allowlist majuscules)
4. Assemblage des lettres

In [ ]:
from utils.page1_parser import read_firstname, read_name

if pdf_files:
    roi_prenom = get_roi(pdf_page1, ROI_FIRSTNAME)
    roi_nom    = get_roi(pdf_page1, ROI_NAME)

    fig, axes = plt.subplots(2, 2, figsize=(14, 4))

    axes[0, 0].imshow(cv2.cvtColor(roi_prenom, cv2.COLOR_BGR2RGB))
    axes[0, 0].set_title('ROI Prénom (brut)')
    axes[0, 0].axis('off')

    # Projection verticale
    gray_p = _to_gray(roi_prenom)
    _, bin_p = cv2.threshold(gray_p, 200, 255, cv2.THRESH_BINARY_INV)
    col_proj = np.sum(bin_p.astype(np.float32), axis=0) / (bin_p.shape[0] * 255)
    axes[0, 1].bar(range(len(col_proj)), col_proj, color='steelblue', width=1)
    axes[0, 1].axhline(0.30, color='red', linestyle='--', label='Seuil séparateur')
    axes[0, 1].set_title('Projection verticale (séparateurs)')
    axes[0, 1].legend()

    axes[1, 0].imshow(cv2.cvtColor(roi_nom, cv2.COLOR_BGR2RGB))
    axes[1, 0].set_title('ROI Nom (brut)')
    axes[1, 0].axis('off')

    gray_n = _to_gray(roi_nom)
    _, bin_n = cv2.threshold(gray_n, 200, 255, cv2.THRESH_BINARY_INV)
    col_proj_n = np.sum(bin_n.astype(np.float32), axis=0) / (bin_n.shape[0] * 255)
    axes[1, 1].bar(range(len(col_proj_n)), col_proj_n, color='darkorange', width=1)
    axes[1, 1].axhline(0.30, color='red', linestyle='--', label='Seuil séparateur')
    axes[1, 1].set_title('Projection verticale Nom')
    axes[1, 1].legend()

    plt.tight_layout()
    plt.show()

    # Lire le prénom et le nom
    prenom = read_firstname(pdf_page1)
    nom    = read_name(pdf_page1)
    print(f'Prénom lu : "{prenom}"')
    print(f'Nom lu    : "{nom}"')

## 4. Lecture des réponses numériques (mantisse × 10^exposant)

Les réponses numériques manuscrites sont au format scientifique :
- **Mantisse** : valeur décimale (ex: 3.25)
- **Exposant** : entier (ex: -1)
- **Unité** : texte (ex: 'm/s')

In [ ]:
from utils.exam_parser import (
    detect_question_blocks, parse_question_block, EXAM_START_PAGE
)

EXAM_START = 4  # page 5 = index 4

if pdf_files and len(pages) > EXAM_START:
    # Chercher des blocs avec réponses numériques
    found_numerical = False
    for page_idx in range(EXAM_START, min(EXAM_START + 3, len(pages))):
        exam_page = normalize_page(pages[page_idx])
        blocks = detect_question_blocks(exam_page)

        for y_start, y_end in blocks:
            block = exam_page[y_start:y_end, :]
            if block.shape[0] < 150:
                continue

            header_cut = max(20, int(block.shape[0] * 0.22))
            content = block[header_cut:, :]

            from utils.exam_parser import _has_numerical_answer, _parse_numerical_answer
            if _has_numerical_answer(content):
                result = _parse_numerical_answer(content)
                print(f'  Mantisse={result["mantisse"]}, Exposant={result["exposant"]}, Unité={result["unite"]}')

                # Visualiser les zones mantisse/exposant/unité
                from utils.exam_parser import _find_answer_boxes
                boxes = _find_answer_boxes(content)
                vis_num = content.copy()
                colors = [(0, 255, 0), (0, 0, 255), (255, 0, 0)]
                for bi, (bx, by, bw, bh) in enumerate(boxes[:3]):
                    cv2.rectangle(vis_num, (bx, by), (bx+bw, by+bh), colors[bi % 3], 2)

                fig, axes = plt.subplots(1, 2, figsize=(12, 4))
                axes[0].imshow(cv2.cvtColor(content, cv2.COLOR_BGR2RGB))
                axes[0].set_title('Contenu du bloc numérique')
                axes[0].axis('off')

                axes[1].imshow(cv2.cvtColor(vis_num, cv2.COLOR_BGR2RGB))
                axes[1].set_title(f'Zones détectées: M={result["mantisse"]}, E={result["exposant"]}')
                axes[1].axis('off')

                plt.tight_layout()
                plt.show()
                found_numerical = True
                break
        if found_numerical:
            break

    if not found_numerical:
        print('Aucun bloc numérique trouvé dans les premières pages d\'examen')

## 5. Évaluation de l'OCR sur la base vérité

Comparaison des résultats OCR avec les vérités terrain des fichiers `.xlsx`.

In [ ]:
import pandas as pd
from utils.page1_parser import parse_page1

xlsx_files = sorted([f for f in os.listdir(FORM_DIR) if f.lower().endswith('.xlsx')])

results = []
n_eval = min(5, len(pdf_files))

for fname in pdf_files[:n_eval]:
    sid = fname.replace('.pdf', '').split('_')[-1]
    xlsx_name = f'EXAM_FORM1_{sid}.xlsx'
    xlsx_path = os.path.join(FORM_DIR, xlsx_name)

    if not os.path.isfile(xlsx_path):
        continue

    # Lire la vérité terrain
    try:
        df = pd.read_excel(xlsx_path, sheet_name='PAGE-01', header=None)
        gt = dict(zip(df[0].astype(str), df[1].astype(str)))
    except Exception:
        continue

    # Lire via OCR
    pages_i = pdf_to_images(os.path.join(FORM_DIR, fname), dpi=150)
    pred = parse_page1(pages_i[0], desc_db=None)

    results.append({
        'StudentID': sid,
        'Module GT': gt.get('Module', ''),
        'Module Pred': pred.get('module', ''),
        'Note max GT': gt.get('Note maximale', ''),
        'Note max Pred': str(pred.get('note_max', '')),
        'Prénom GT': gt.get('Prénom', ''),
        'Prénom Pred': pred.get('prenom', ''),
    })

if results:
    df_eval = pd.DataFrame(results)
    print(df_eval.to_string(index=False))
else:
    print('Pas de données pour l\'évaluation')

## Bilan

| Tâche OCR | Méthode | Notes |
|---|---|---|
| CODES EXAM (imprimé) | EasyOCR + regex | Bonne précision sur PDF |
| Notes (chiffres imprimés) | EasyOCR + segmentation bas niveau | Fallback disponible |
| Prénom/Nom (manuscrit) | Segmentation cellules + EasyOCR | Sensible à la qualité écriture |
| Mantisse (manuscrit) | CLAHE + EasyOCR + segmentation CC | Complexe avec encre rouge |
| Exposant (manuscrit) | Segmentation + classificateur | |

**Prochaine étape** → `06_programme1_presences.ipynb` : pipeline complet de validation des présences.